In [ ]:
# Python — Google Colab
# ============================================================
# CELL 1 — Instalasi Library
# Jalankan cell ini SEKALI di awal sesi baru
# Waktu instalasi: 2–5 menit
# ============================================================

# Install library utama
!pip install earthengine-api --quiet   # Google Earth Engine Python API
!pip install geemap --quiet            # Wrapper interaktif untuk GEE

# Verifikasi instalasi
import ee
import geemap
print(f"earthengine-api versi: {ee.__version__}")
print(f"geemap versi: {geemap.__version__}")
print("✅ Instalasi berhasil!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.6 MB/s eta 0:00:00
earthengine-api versi: 1.7.39
geemap versi: 0.38.3
✅ Instalasi berhasil!


In [ ]:
# CELL 2. Autentifikasi dan Inisialisasi GEE
#==========================================

import ee
import geemap

# Langkah 1: Autentikasi
ee.Authenticate()

# Langkah 2: Inisialisasi
ee.Initialize(project="latihan-pemograman-spasial")

#Verifikasi
print("Status koneksi GEE:", ee.String("Berhasil terhubung!").getInfo())
print("✅ GEE siap digunakan!")

Status koneksi GEE: Berhasil terhubung!
✅ GEE siap digunakan!


In [ ]:
# CELL 3. Import Semua Library yang Dibutuhkan
#=============================================

import ee          # Google Earth Engine API
import geemap      # Peta interaktif berbasis GEE
import json        # Untuk parsing JSON (nanti digunakan untuk AOI)

print("📦 Semua library berhasil diimport!")


📦 Semua library berhasil diimport!


In [ ]:
# CELL 4. Membuat Peta Interaktif dan Menampilkan Citra Landsat
#=============================================================

# BUAT OBEJK PETA INTERAKTIF GEEMAP
## --- geemap.Map() untuk menggabungkan GEE dengan visualisasi Leaflet.js
Map = geemap.Map()

# MENENTUKAN AREA KAJIAN (Parepare)
# Format: ee.Geometry.Point([longitude, latitude])
aoi_point = ee.Geometry.Point([112.7123, -7.308])

# LOAD ImageCollection LANDSAT 9
landsat_collection = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2") \
    .filterDate("2025-01-01", "2025-12-31") \
    .filterBounds(aoi_point) \
    .filterMetadata("CLOUD_COVER", "less_than", 10)

## --- Cek berapa banyak citra tersedia
jumlah_citra = landsat_collection.size().getInfo()
print(f"📊 Jumlah citra tersedia: {jumlah_citra} scene")

## --- Ambil citra dengan cloud cover terendah (kualitas terbaik)
citra_terbaik = landsat_collection \
    .sort("CLOUD_COVER") \
    .first()

## --- Tampilkan tanggal akuisisi
tanggal = citra_terbaik.date().format("YYYY-MM-dd").getInfo()
print(f"📅 Tanggal akuisisi citra terbaik: {tanggal}")

# --- Parameter visualisasi True Color
vis_true = {
    "bands": ["SR_B4", "SR_B3", "SR_B2"],
    "min": 7000,
    "max": 13000,
    "gamma": 1.4
}

## --- Parameter visualisasi False Color (NIR-Red-Green)
vis_false = {
    "bands": ["SR_B5", "SR_B4", "SR_B3"],
    "min": 7000,
    "max": 20000
}
## --- Tambahkan layer ke peta
Map.centerObject(aoi_point, 10)
Map.addLayer(citra_terbaik, vis_true,  "Landsat9 True Color")
Map.addLayer(citra_terbaik, vis_false, "Landsat9 False Color", shown=False)
Map.addLayer(aoi_point, {"color": "FF0000"}, "Titik AOI", shown=True)

# TAMPILKAN PETA
Map



📊 Jumlah citra tersedia: 4 scene
📅 Tanggal akuisisi citra terbaik: 2025-08-06


Map(center=[-7.307999999999999, 112.7123], controls=(WidgetControl(options=['position', 'transparent_bg'], pos…

In [ ]:
# CELL 5. Menampilkan Metadata Citra
#===================================\

# AMBIL SEMUA PROPERTI (METADATA) CITRA
info_citra = citra_terbaik.getInfo()

#MENAMPILKAN INFORMASI PENTING
properties = citra_terbaik.toDictionary()

print("=" * 50)
print("INFORMASI CITRA LANDSAT 9")
print("=" * 50)

cloud_cover = citra_terbaik.get("CLOUD_COVER").getInfo()
spacecraft  = citra_terbaik.get("SPACECRAFT_ID").getInfo()
path        = citra_terbaik.get("WRS_PATH").getInfo()
row         = citra_terbaik.get("WRS_ROW").getInfo()

print(f"Satelit      : {spacecraft}")
print(f"Tanggal      : {tanggal}")
print(f"Path/Row     : {path}/{row}")
print(f"Cloud Cover  : {cloud_cover:.1f}%")

# MENAMPILKAN NAMA BAND
band_names = citra_terbaik.bandNames().getInfo()
print(f"Band tersedia: {band_names}")
print("=" * 50)



INFORMASI CITRA LANDSAT 9
Satelit      : LANDSAT_9
Tanggal      : 2025-08-06
Path/Row     : 118/65
Cloud Cover  : 1.6%
Band tersedia: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


In [ ]:
# CELL 6. Sentinel-2 Sebagai Alternatif (resolusi 10m)
#=====================================================

# SENTINEL-2 SURFACE REFLECTANCE--RESOLUSI LEBIH TINGGI (10 vs 30)
sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterDate("2025-01-01", "2025-12-31") \
    .filterBounds(aoi_point) \
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10)) \
    .sort("CLOUDY_PIXEL_PERCENTAGE") \
    .first()

## --- Cek Tanggal
tgl_s2 = sentinel2.date().format("YYYY-MM-dd").getInfo()
cloud_s2 = sentinel2.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()
print(f"Sentinel-2 terbaik: {tgl_s2} (cloud: {cloud_s2:.1f}%)")

## --- Visualisasi Sentinel-2 True Color
vis_s2_true = {
    "bands": ["B4", "B3", "B2"],  # Band S2 berbeda dari Landsat!
    "min": 0,
    "max": 3000
}

## --- Menambahkan ke peta yang sama
Map.addLayer(sentinel2, vis_s2_true, "Sentinel-2 True Color", shown=False)
Map  # Tampilkan ulang peta dengan layer baru


Sentinel-2 terbaik: 2025-07-21 (cloud: 0.1%)


Map(center=[-7.307999999999999, 112.7123], controls=(WidgetControl(options=['position', 'transparent_bg'], pos…

In [ ]:
# CELL 7. Mendefisikan dan Menyimpan AOI Proyek
#==============================================

import ee
import geemap
import json

# Opsi A. AOI menggunakan koordinat manual
aoi_coords = [
    [112.34216013619466,-7.013376002953298],
    [112.34216013619466,-7.672594006463851],
    [113.06725779244466,-7.699813058036462],
    [113.06725779244466,-7.024280000073087],
    [112.34216013619466,-7.013376002953298]
]

aoi = ee.Geometry.Polygon([aoi_coords]) # Buat Objek Geometry GEE

# Opsi B. AOI dari Bounding box
# format = aoi = ee.Geometry.Rectangle([lon_barat, lat_selatan, lon_timur, lat_utara])
aoi = ee.Geometry.Rectangle([112.24252647646517,-7.805844136513645, 113.21481651552767, -6.90148870185857])


# --- Informasi AOI
luas_m2  = aoi.area(maxError=1).getInfo()
luas_km2 = luas_m2 / 1_000_000
print(f"📍 AOI berhasil didefinisikan")
print(f"📐 Luas AOI: {luas_km2:.2f} km²")


# --- Visualisasi AOI di Peta
Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(
    aoi,
    {"color": "FF0000", "fillColor": "FF000020"},
    "AOI Proyek"
)
Map



📍 AOI berhasil didefinisikan
📐 Luas AOI: 10782.62 km²


Map(center=[-7.353774683266315, 112.72867149599635], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# Cell 8. Simpan AOI sebagai GeoJSON di GDrive
#=============================================

from google.colab import drive
import json, os

# Mount GDrive
drive.mount('/content/drive')

# Buat Folder di GDrive
folder_proyek = "/content/drive/MyDrive/Colab Notebooks/ProyekGEE_PemSpasial"
os.makedirs(folder_proyek, exist_ok=True)
os.makedirs(f"{folder_proyek}/data_aoi", exist_ok=True)

# Konversi AOI ke GeoJSON
aoi_geojson = aoi.getInfo()  # ambil representasi Python dict

# Menjadi  FeatureCollection GeoJSON Standar
geojson_output = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {
            "nama": "AOI_Proyek_PemSpasial",
            "mahasiswa": "Nur Faatihah Ashar",
            "nim": "V126241002",
            "deskripsi": "Area kajian surabaya proyek Praktek Pemrograman Spasial 2026"
        },
        "geometry": aoi_geojson
    }]
}

# Simpan ke Drive
path_output = f"{folder_proyek}/data_aoi/AOI_proyek.geojson"
with open(path_output, "w") as f:
    json.dump(geojson_output, f, indent=2)

print(f"✅ AOI berhasil disimpan: {path_output}")
print(f"📁 Ukuran file: {os.path.getsize(path_output)} bytes")



Mounted at /content/drive
✅ AOI berhasil disimpan: /content/drive/MyDrive/Colab Notebooks/ProyekGEE_PemSpasial/data_aoi/AOI_proyek.geojson
📁 Ukuran file: 910 bytes


In [ ]:
# Cell 9. Upload AOI ke GEE Assets
#=================================

# Mengaupload menggunakan geemap
# geemap.shp_to_ee() untuk shapefile
# atau export via ee.batch.Export untuk dari Colab

# Export AOI sebagai FeatureCollection ke GEE Assets
aoi_feature = ee.Feature(aoi, {
    "nama": "AOI_Proyek",
    "mahasiswa": "Nur Faatihah Ashar",
    "nim": "V126241002"
})

aoi_fc = ee.FeatureCollection([aoi_feature])

# Ganti "your_username" dengan username GEE Anda
task = ee.batch.Export.table.toAsset(
    collection=aoi_fc,
    description="Upload_AOI_Proyek",
    assetId="projects/earthengine-legacy/assets/users/fatihahaashar/AOI_PemSpasial"
)

task.start()
print("🚀 Task upload AOI dimulai!")
print("Cek status di GEE Code Editor → Tasks")

# Monitor status task
import time
status = task.status()
print(f"Status awal: {status['state']}")


🚀 Task upload AOI dimulai!
Cek status di GEE Code Editor → Tasks
Status awal: READY


In [ ]:
# Cell 10. Koneksi Colab ke GitHub

# Konfigurasi identitas git
!git config --global user.email "fatihahaashar@gmail.com"
!git config --global user.name "Nur Faatihah Ashar"

# Clone repository dari GitHub ke Colab
# Ganti URL dengan URL repository GitHub Anda
!git clone https://github.com/if44s/PemSpasial_GEE_NIM.git

# Masuk ke folder repository
%cd PemSpasial_GEE_V126241002

# Buat struktur folder
!mkdir -p data_aoi data_raw data_processed notebooks gee_scripts outputs/maps outputs/stats outputs/charts report

# Salin notebook dari lokasi Colab ke folder repository
!cp /content/drive/MyDrive/Colab Notebooks/ProyekGEE_PemSpasial/data_aoi/AOI_proyek.geojson data_aoi/

# Tambahkan file ke git dan commit
!git add .
!git commit -m "Pertemuan 1: Setup project structure, AOI defined"

# Push ke GitHub (perlu personal access token)
# Ganti URL dengan URL repo Anda dan masukkan token jika diminta
!git push origin main

print("✅ Commit pertama berhasil!")


Cloning into 'PemSpasial_GEE_NIM'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'PemSpasial_GEE_V126241002'
/content
cp: cannot stat '/content/drive/MyDrive/Colab': No such file or directory
cp: cannot stat 'Notebooks/ProyekGEE_PemSpasial/data_aoi/AOI_proyek.geojson': No such file or directory
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
✅ Commit pertama berhasil!


Template Jurnal Harian
# ============================================================
# CELL 11 — Jurnal Harian Pertemuan 1
# ============================================================

---
## 📔 Jurnal Harian — Pertemuan 1
**Tanggal**: 3 September 2026

**Nama**: Nur Faatihah Ashar

### ✅ Apa yang berhasil hari ini?
Cell 1 hingga Cell 10 berhasil dijalanka. Saya mulai sedikit memahami bahasa pemograman dasar, mulai dari menjalankan script, membuat AOI, hingga menampilkannya di peta. Output yang paling menarik adalah visualisasi citra Sentinel 2 Surabaya dan hasil pemfilteran citra berdasarkan tanggal, lokasi, dan tutupan awan.

### ❌ Apa yang tidak berjalan atau membingungkan?
Kendala terjadi pada Cell 8 dan Cell 10 saat menjalankan script. Salah satu error yang muncul adalah "Invalid GeoJSON geometry" pada bagian filterBounds(). Awalnya koordinat lokasi dimasukkan langsung dalam bentuk [longitude, latitude], sehingga tidak dikenali sebagai geometry oleh Google Earth Engine. Error tersebut diatasi dengan mengubah koordinat menjadi ee.Geometry.Point(). Kemudian citra sentinel 2 pada google colab yang terload tidak menutupi keseluruhan AOI Kota Surabaya.

### 💡 Keputusan teknis yang dibuat:
Saya memilih Kota Surabaya karena memiliki wilayah yang relatif datar dan kawasan terbangun yang cukup padat. Kondisi tersebut dapat meningkatkan potensi terjadinya genangan atau banjir, terutama saat curah hujan tinggi.Untuk data citra, saya memilih Sentinel-2 karena memiliki resolusi spasial yang cukup tinggi, yaitu hingga 10 meter pada beberapa band, sehingga dapat memberikan detail permukaan wilayah yang lebih baik. Saya menggunakan filter cloud cover kurang dari 10% agar citra yang dipilih memiliki tutupan awan yang relatif rendah dan lebih mudah digunakan untuk analisis.

### 🔗 Koneksi dengan proyek ke depan:
Setup hari ini bakal jadi dasar buat analisis di pertemuan berikutnya. AOI Surabaya dan citra Sentinel-2 yang sudah dipilih nantinya akan digunakan untuk proses analisis. Sebelum lanjut, saya perlu memastikan AOI dan citra yang digunakan sudah sesuai serta menyiapkan data dan script untuk tahap berikutnya.

### ⭐ Hal paling menarik dari pertemuan ini:
Bagaimana saya mulai belajar bahasa pemograman walaupun memiliki dasar bahasa pemograman sebelumnya, tapi seruu walau agak pusing


---
